<div class="alert alert-block alert-info">
<b>Submission for feedback:</b> 3 November 2026 (Tuesday), 23:00 (Europe/Berlin), via ILIAS.
</div>

# Exercise 1.1. Train a multilayer perceptron (MLP) with pytorch.

The goal of this exercise is to get familiar with the basics of PyTorch and train a multilayer perceptron (MLP) model.

If you are not familiar with PyTorch, there is a number of good tutorials [here](https://pytorch.org/tutorials/index.html). We recommend the following ones:
* [What is PyTorch?](https://pytorch.org/tutorials/beginner/blitz/tensor_tutorial.html#sphx-glr-beginner-blitz-tensor-tutorial-py)
* [Autograd: Automatic Differentiation](https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#sphx-glr-beginner-blitz-autograd-tutorial-py)
* [Learning PyTorch with Examples](https://pytorch.org/tutorials/beginner/pytorch_with_examples.html)
* [Neural Networks](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py)

In [ ]:
# False:正常训练。
# True:跳过训练,直接加载已保存好的权重(用于评分或节省时间)
skip_training = False  # Set this flag to True before validation and submission

In [ ]:
# During evaluation, this cell sets skip_training to True
# skip_training = True

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F

import tools

In [ ]:
# Select device which you are going to use for training
#device = torch.device("cuda:0")
device = torch.device("cpu")

In [ ]:
if skip_training:
    # The models are always evaluated on CPU
    device = torch.device("cpu")

# Data

We will train the MLP on a toy regression problem.

In [ ]:
# Let us generate random toy data
def get_data():
    pass
    '''
    1. 生成输入 x
    '''
    # 固定随机种子,保证每次运行结果一致(可复现)。
    np.random.seed(2)
    # 从标准正态分布 N(0,1) 采样 100 个点,形状为 (100, 1),即 100 个样本、每个样本 1 个特征。
    x = np.random.randn(100, 1)
    # 按升序排序,这样后面画折线或观察趋势时 x 轴是单调的。
    x = np.sort(x, axis=0)

    '''
    2. 生成目标值 targets
    '''
    # 真实函数是 sin(2πx/3),周期为 3
    targets = np.sin(x * 2 * np.pi / 3)
    # 加上标准差为 0.2 的高斯噪声,模拟真实观测。
    # *targets.shape 把 (100, 1) 解包成参数,生成同形状噪声。
    targets = targets + 0.2 * np.random.randn(*targets.shape)

    # Convert to PyTorch tensors
    '''
    3.  转为 PyTorch 张量
    '''
    x = torch.FloatTensor(x)
    targets = torch.FloatTensor(targets)
    
    return x, targets

# 调用get_data函数
x, targets = get_data()
# Plot the data
'''
4.  画图
用 matplotlib 创建一张画布:
fig 是整张图(Figure),可以用来设置大小、保存图片等。
ax 是图里的坐标系(Axes),真正的画图操作都在它上面进行。
参数 1 表示只要 1 个子图(1 行 1 列)。
在坐标系上画图:横轴用 x,纵轴用 targets。
第三个参数 '.' 是格式字符串,表示每个数据点只画成一个小圆点,不用线连起来
'''
fig, ax = plt.subplots(1)
ax.plot(x, targets, '.')

# Multilayer perceptron (MLP) network with two hidden layers

We will create a simple multilayer perceptron (MLP) network. The model has
- input dimensionality 1
- one hidden layer with 10 units with Tanh nonlinearity
- one hidden layer with 11 units with Tanh nonlinearity
- linear output layer with output dimensionality 1 and no nonlinearity.

Hints:
* You may want to look at [this tutorial](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py) for reference.
* You can use [`nn.Linear`](https://pytorch.org/docs/stable/nn.html?highlight=nn%20linear#torch.nn.Linear)
module to define the fully-connected layers of the MLP.
* Simple architectures are usually created using module [`torch.nn.Sequential`](https://pytorch.org/docs/stable/nn.html#torch.nn.Sequential). You do not have to use this module in this exercise.

搭建网络结构是 1 → 10 → 11 → 1的MLP model,两个隐藏层都用 Tanh,输出层是线性的。

In [ ]:
'''
结构：输入 (1) → Linear(1, 10) → Tanh → Linear(10, 11) → Tanh → Linear(11, 1) → 输出 (1)
输入维度 1:因为 x 只有一个特征,所以输入形状是 (N, 1)(这就是前面要保留 (100, 1) 形状的原因)。
第一隐藏层 10 个单元,激活函数 Tanh。
第二隐藏层 11 个单元,激活函数 Tanh。注意它的输入是 10 维,输出是 11 维。
输出层 1 个单元,不加激活函数,因为这是回归任务,输出需要能取任意实数,而sigmoid会输出概率。

用 nn.Sequential 的写法(最简洁，此处不用)
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(1, 10),   # 输入 1 维 -> 隐藏层 1(10 个单元)
    nn.Tanh(),
    nn.Linear(10, 11),  # 隐藏层 1 -> 隐藏层 2(11 个单元)
    nn.Tanh(),
    nn.Linear(11, 1),   # 隐藏层 2 -> 输出(1 维,无非线性)
)
'''

In [ ]:
class MLP(nn.Module):# extends from nn.Module
    def __init__(self, n_inputs=1):# 声明有哪些层
        # YOUR CODE HERE
        # 必须先调用父类nn.Module的构造函数,否则后面给 self.fc1 赋值时,nn.Module 无法注册子模块,会报错
        super().__init__()
        # 三个全连接层,每层计算 y = xWᵀ + b。
        # x.shape = (N, in_features),每个样本是一行
        # Wᵀ = nn.Linear(in_features, out_features) （随机数初始化）
        # b.shape = (out_features,),通过广播加到每一行（随机数初始化）
        self.fc1 = nn.Linear(1, 10)
        self.fc2 = nn.Linear(10, 11)
        self.fc3 = nn.Linear(11, 1)
        self.act = nn.Tanh() # 隐藏层

    def forward(self, x):
        """
        Args:
          x of shape (n_samples, n_inputs): Model inputs.
        
        Returns:
          y of shape (n_samples, 1): Model outputs.
        """
        # YOUR CODE HERE
        x = self.act(self.fc1(x)) # (N,1) * (1,10)  -> (N,10)
        x = self.act(self.fc2(x)) # (N,10) * (10,11) -> (N,11)
        return self.fc3(x) # (N,11) * (11,1) -> (N,1)

后面`model.parameters()` 里的参数,来自你在 `MLP` 类里定义的那三个 `nn.Linear` 层。

## 参数是怎么产生的

```python
self.fc1 = nn.Linear(1, 10)
```
创建 `nn.Linear` 时,PyTorch 会在内部自动创建两个张量,并用随机数初始化:

- `weight`:形状 `(10, 1)`,即 10 个权重
- `bias`:形状 `(10,)`,即 10 个偏置

这两个张量被包装成 `nn.Parameter` 类型。`nn.Parameter` 是一种特殊的张量,默认 `requires_grad=True`,表示"这是需要学习的量,要计算它的梯度"。

In [ ]:
# Let us create the network and make sure it can process a random input of the right shape
# 这是一个形状冒烟测试(shape sanity test):不关心输出数值对不对,只检查网络能否接收正确形状的输入,并产生正确形状的输出。发现在编写MLP时候的错误（如层与层维度没衔接）
def test_MLP_shapes():
    n_inputs = 1
    n_samples = 10
    net = MLP()
    # torch.randn(10, 1) 生成一个形状 (10, 1) 的随机输入(标准正态),相当于 10 个样本、每个 1 个特征。数据流是:
    # (10, 1) → fc1 → (10, 10) → Tanh → fc2 → (10, 11) → Tanh → fc3 → (10, 1)
    y = net(torch.randn(n_samples, n_inputs))
    assert y.shape == torch.Size([n_samples, 1]), f"Bad y.shape={y.shape}"
    print('Success')

test_MLP_shapes()

## Train the MLP network

### Training loop

Your task is to implement the training loop.
You may find it useful to look at [this tutorial](https://pytorch.org/tutorials/beginner/blitz/neural_networks_tutorial.html#sphx-glr-beginner-blitz-neural-networks-tutorial-py).
You should perform the following steps:
* Set all gradient values to zero.
* Calculate the output of the model for all training examples.
* Calculate the MSE loss (see [`mse_loss`](https://pytorch.org/docs/stable/nn.functional.html?highlight=mse_loss#torch.nn.functional.mse_loss)).
* Backpropagate the gradients: compute the gradients of the loss wrt to all the parameters of the model.
* Update the parameters of the model using the chosen optimizer.


Recommended hyperparameters:
* [Adam optimizer](https://pytorch.org/docs/stable/optim.html#torch.optim.Adam) with learning rate 0.01.
* You can process the data in the full-batch model (computing the gradients using all training data).
* Number of iterations (parameter updates): 200.

<span style="display:none">
Tip: In the training loop, add a comment saying "zero the gradient buffers" after optimizer.zero_grad().
</span>

这一步是**写训练循环**:用 Adam 对整个数据集做全批量训练,迭代 200 次,让 MLP 拟合带噪正弦数据。

## 要求拆解

题目给出的五个步骤正好对应一次迭代里的五行核心代码:

1. 梯度清零:`optimizer.zero_grad()`
2. 前向传播:`y_pred = model(x)`
3. 计算 MSE 损失:`loss = F.mse_loss(y_pred, targets)`
4. 反向传播:`loss.backward()`
5. 更新参数:`optimizer.step()`

超参数: Adam优化器,学习率 0.01,全批量(一次用全部 100 个点),共 200 次更新。

另外有一个格式要求:在 `optimizer.zero_grad()` 之后要加注释 `# zero the gradient buffers`。

In [ ]:
# Create the model
# 实例化你之前写的 MLP,并把所有参数移到 device(CPU 或 GPU)上。
# 必须在创建优化器之前完成,否则优化器拿到的参数引用会出问题
model = MLP()
model.to(device)

In [ ]:
# Implement the training loop here
'''
可以把它想成反复做同一件事:先猜(前向传播),再算猜得有多差(损失),然后找出怎么改能变好(反向传播),用Adam优化器最后改一点点(梯度下降权重参数更新)。重复 200 次,曲线就越来越贴近数据。
'''
if not skip_training:
    # YOUR CODE HERE
    # 创建优化器。model.parameters() 把网络里所有可学习的权重和偏置交给 Adam,让它负责更新;lr=0.01 是学习率,控制每次更新的步子大小
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    # 循环 200 次,每次是一轮"计算误差 → 调整参数"。因为是全批量训练,每次都用全部 100 个数据点,所以 200 次循环就是 200 次参数更新。
    for i in range(200):
        # 把上一轮留下的梯度清零。PyTorch 的 backward() 会把梯度累加,不清零就会混进旧梯度
        optimizer.zero_grad()   # zero the gradient buffers
        # 前向传播:把 100 个 x 送进网络,得到 100 个预测值,形状 (100, 1)
        y_pred = model(x)
        # 计算均方误差:预测值与真实值之差的平方,再取平均。这个数越小,说明拟合得越好。
        loss = F.mse_loss(y_pred, targets)
        # 反向传播:自动算出损失对每个参数的梯度(参数往哪个方向改、损失会变小)。
        loss.backward()
        # 梯度下降：Adam 根据梯度更新model所有参数（权重&bias）。
        # 公式：weight = weight - learning_rate * gradient(如dLoss/dWeight)
        # 这一步之后,model就比上一轮稍微好了一点, 然后进入下个迭代
        optimizer.step()
        # 每 20 次迭代打印一次当前损失,方便观察训练是否在收敛。loss.item() 把张量转成普通数字,:.4f 表示保留 4 位小数。
        if (i + 1) % 20 == 0:
            print(f"iter {i+1}, loss = {loss.item():.4f}")

In [ ]:
# Save the model to disk (the pth-files have to be submitted together with your notebook)
# Set confirm=False if you do not want to be asked for confirmation before saving.
if not skip_training:
    tools.save_model(model, '2_mlp.pth', confirm=True)

In [ ]:
# 跳过训练时,重新创建一个空模型,再把 .pth 文件里的权重读进去。和上一个单元格互补:训练就保存,不训练就加载
if skip_training:
    model = MLP()
    tools.load_model(model, '2_mlp.pth', device)

In [ ]:
# Plot the trained MLP
#训练完成后,把模型的预测曲线和原始数据画在同一张图上,用来直观检查拟合效果。
'''
with进入一个"不记录梯度"的上下文。在这个代码块里做的所有张量运算都不会构建计算图。
因为这里只是用模型做预测,不需要反向传播,所以关掉梯度可以省内存、跑得更快。
另外,带梯度的张量不能直接调用 .numpy(),在这个块里就不会有这个问题
'''
with torch.no_grad():
    fig, ax = plt.subplots(1)
    ax.plot(x, targets, '.')
    y = model(x) # 把全部 100 个输入点 x 送进训练好的 MLP,得到预测值 y,形状 (100, 1)。
    # 画预测曲线。y.numpy() 把 PyTorch 张量转成 NumPy 数组,matplotlib 才能稳定处理。'r-' 是格式字符串:r 是红色(red),- 是实线。
    # ax.grid()给坐标系加上网格线,方便读数和对比
    ax.plot(x, y.numpy(), 'r-')
    ax.grid(True)

In [ ]:
# This cell tests MLP

In [ ]:
# This cell tests MLP